PREDICTING STARTUP SURVIVAL: A MACHINE LEARNING PERSPECTIVE
 
> In questa tesi vengono combinate tecniche di Data Mining e Machine Learning
con l’obiettivo di identificare i principali fattori di rischio per le startup (con un
particolare focus su Competitors, Team e Funding), in modo da prevederne l’eventuale fallimento.

> I dati presi in esame provengono dalla piattaforma PitchBook, e
sono stati elaborati in linea con la letteratura economica recente.

> A partire dalle metriche calcolate, sono stati addestrati diversi modelli (Alberi
Decisionali, Random Forest, Reti Neurali Artificiali), i quali sono stati successivamente valutati in base alla capacità di classificare correttamente le startup fallite.
Questi esperimenti hanno permesso, inoltre, di cogliere i principali indicatori di "sopravvivenza" delle startup, mediante il valore dell’importanza che ogni modello
assegna a una determinata feature. Un ulteriore studio è stato svolto tramite l’impiego del Random Survival Forest, una versione alternativa del Random Forest
ideata per l’analisi della sopravvivenza.

In [1]:
%load_ext autoreload
%autoreload 2

import polars as pl


Temporizzazione Competitors

In [2]:
# ============================================================
# 1. LETTURA DEI TRE FILE
# ============================================================
companies = pl.read_csv("data/Company.csv", null_values=["NA"])
competitors_raw = pl.read_csv("data/CompanySimilarRelation.csv", null_values=["NA"])
panel = pl.read_csv("data/db_master_panel.csv.gz", null_values=["NA"])


In [3]:


# ============================================================
# 2. CALCOLO YearFounded e MaxYear PER OGNI AZIENDA
# ============================================================
date_cols = [
    "CompanyFinancingStatusDate",
    "BusinessStatusDate",
    "OwnershipStatusDate",
    "FirstFinancingDate",
    "LastKnownValuationDate",
]

for col in date_cols:
    if col in companies.columns:
        companies = companies.with_columns(
            pl.col(col)
            .str.to_date("%m/%d/%Y", strict=False)
            .fill_null(pl.col(col).str.to_date("%m/%d/%y", strict=False))
            .alias(col)
        )

if "FiscalPeriod" in companies.columns:
    companies = (
        companies
        .with_columns(
            pl.col("FiscalPeriod").str.extract(r"(\d{4})$").cast(pl.Int32, strict=False).alias("_fy"),
            pl.col("FiscalPeriod").str.extract(r"(\d)Q").cast(pl.Int32, strict=False).alias("_fq"),
        )
        .with_columns(
            pl.when(pl.col("_fq") == 1).then(3)
            .when(pl.col("_fq") == 2).then(6)
            .when(pl.col("_fq") == 3).then(9)
            .when(pl.col("_fq") == 4).then(12)
            .otherwise(None)
            .alias("_fm")
        )
        .with_columns(pl.date(pl.col("_fy"), pl.col("_fm"), 28).alias("FiscalDate"))
        .drop("_fy", "_fq", "_fm")
    )
else:
    companies = companies.with_columns(pl.lit(None).cast(pl.Date).alias("FiscalDate"))

all_date_cols = [c for c in date_cols + ["FiscalDate"] if c in companies.columns]
year_aliases = [f"_y_{c}" for c in all_date_cols]

companies = companies.with_columns([
    pl.col(c).dt.year().cast(pl.Int32, strict=False).alias(f"_y_{c}")
    for c in all_date_cols
])

companies = companies.with_columns(
    pl.max_horizontal([pl.col(y) for y in year_aliases]).alias("MaxYear"),
    pl.col("YearFounded").cast(pl.Int32, strict=False),
)

# Lookup: CompanyID -> YearFounded, MaxYear, HQCountry
company_life = (
    companies
    .select("CompanyID", "YearFounded", "MaxYear", "HQCountry")
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
)

# ============================================================
# 3. FILTRA E ARRICCHISCI
# ============================================================
# Qui usiamo direttamente competitors_raw: la relazione è letta
# come "CompanyID dichiara SimilarCompanyID come sua simile/competitor".
# Non aggiungiamo la direzione inversa.

valid_panel_ids = panel.select("CompanyID").unique().to_series()
valid_company_ids = company_life.select("CompanyID").unique().to_series()

# --- 3a. COMPETITOR (IsCompetitor == "Yes") ---
comp = (
    competitors_raw
    .filter(
        (pl.col("IsCompetitor") == "Yes") &
        pl.col("CompanyID").is_in(valid_panel_ids) &
        pl.col("SimilarCompanyID").is_in(valid_company_ids)
    )
    .select("CompanyID", "SimilarCompanyID", "SimilarityScore")
    # Finestra di vita del competitor
    .join(
        company_life.select(
            pl.col("CompanyID").alias("SimilarCompanyID"),
            pl.col("YearFounded").alias("YF_comp"),
            pl.col("MaxYear").alias("MY_comp"),
            pl.col("HQCountry").alias("HQCountry_comp"),
        ),
        on="SimilarCompanyID",
        how="inner",
    )
    # Nazione dell'azienda del panel
    .join(
        company_life.select("CompanyID", pl.col("HQCountry").alias("HQCountry_panel")),
        on="CompanyID",
        how="left",
    )
    .with_columns(
        # null se una delle due HQCountry manca: escluso dal conteggio, non contato come False
        (pl.col("HQCountry_panel") == pl.col("HQCountry_comp")).alias("SameCountry")
    )
    .drop("HQCountry_panel", "HQCountry_comp")
)

print(f"Coppie competitor filtrate: {comp.height}")

# --- 3b. TUTTE LE AZIENDE SIMILI (per media SimilarityScore) ---
similar_all = (
    competitors_raw
    .filter(
        pl.col("CompanyID").is_in(valid_panel_ids) &
        pl.col("SimilarCompanyID").is_in(valid_company_ids)
    )
    .select("CompanyID", "SimilarCompanyID", "SimilarityScore")
    .join(
        company_life.select(
            pl.col("CompanyID").alias("SimilarCompanyID"),
            pl.col("YearFounded").alias("YF_sim"),
            pl.col("MaxYear").alias("MY_sim"),
        ),
        on="SimilarCompanyID",
        how="inner",
    )
)

print(f"Coppie simili totali: {similar_all.height}")

# ============================================================
# 4. JOIN CON INEQUALITY
# ============================================================
panel_years = panel.select("CompanyID", "Year_Delta").unique()

# 4a. Competitor attivi
matched_comp = panel_years.join_where(
    comp,
    pl.col("CompanyID") == pl.col("CompanyID_right"),
    pl.col("Year_Delta") >= pl.col("YF_comp"),
    pl.col("Year_Delta") <= pl.col("MY_comp"),
)

# 4b. Tutte le aziende simili attive
matched_sim = panel_years.join_where(
    similar_all,
    pl.col("CompanyID") == pl.col("CompanyID_right"),
    pl.col("Year_Delta") >= pl.col("YF_sim"),
    pl.col("Year_Delta") <= pl.col("MY_sim"),
)

# ============================================================
# 5. AGGREGA PER AZIENDA-ANNO
# ============================================================
competitor_stats = (
    matched_comp
    .group_by("CompanyID", "Year_Delta")
    .agg(
        pl.col("SimilarCompanyID").n_unique().alias("N_ActiveCompetitors"),
        pl.col("SameCountry").drop_nulls().sum().cast(pl.Int64).alias("N_ActiveCompetitors_SameCountry"),
    )
)

similarity_stats = (
    matched_sim
    .group_by("CompanyID", "Year_Delta")
    .agg(
        pl.col("SimilarityScore").mean().alias("Mean_SimilarityScore"),
    )
)

# ============================================================
# 6. UNISCI AL PANEL
# ============================================================
panel = (
    panel
    .join(competitor_stats, on=["CompanyID", "Year_Delta"], how="left")
    .join(similarity_stats, on=["CompanyID", "Year_Delta"], how="left")
    .with_columns(
        pl.col("N_ActiveCompetitors").fill_null(0),
        pl.col("N_ActiveCompetitors_SameCountry").fill_null(0),
        pl.col("Mean_SimilarityScore").fill_null(0.0),
    )
)

# ============================================================
# 7. COMPETITOR STATS SENZA FINESTRA TEMPORALE (per esperimento nowindow)
# Conta tutti i competitor dichiarati indipendentemente dalla loro finestra di vita.
# ============================================================
nowindow_competitor_stats = (
    comp
    .group_by("CompanyID")
    .agg(
        pl.col("SimilarCompanyID").n_unique().alias("N_AllCompetitors"),
        pl.col("SameCountry").drop_nulls().sum().cast(pl.Int64).alias("N_AllCompetitors_SameCountry"),
    )
)

nowindow_similarity_stats = (
    similar_all
    .group_by("CompanyID")
    .agg(
        pl.col("SimilarityScore").mean().alias("Mean_AllSimilarityScore"),
    )
)

temporal_features=panel.select(
        "CompanyID", "Year_Delta",
        "N_ActiveCompetitors","N_ActiveCompetitors_SameCountry", 
        "Mean_SimilarityScore"
    )



/tmp/ipykernel_122899/1815676774.py:75: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(


Coppie competitor filtrate: 25690


/tmp/ipykernel_122899/1815676774.py:110: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(


Coppie simili totali: 185009


Creazione dataset finale

In [4]:
df = pl.read_csv("data/db_master_panel.csv.gz", null_values=["NA"])

df = (
    df
    .drop("N_Competitors", "Same_Country", "SimilarityScoreMean")
    .join(temporal_features, on=["CompanyID", "Year_Delta"], how="left")
    .rename({
        "N_ActiveCompetitors": "N_Competitors",
        "N_ActiveCompetitors_SameCountry": "Same_Country",
        "Mean_SimilarityScore": "SimilarityScoreMean",
    })
)

df = df.drop("N_Europe", "N_Outside_Europe")

# Aggiunge le versioni nowindow (tutti i competitor, senza filtro temporale)
df = (
    df
    .join(nowindow_competitor_stats, on="CompanyID", how="left")
    .join(nowindow_similarity_stats, on="CompanyID", how="left")
    .rename({
        "N_AllCompetitors": "N_Competitors_All",
        "N_AllCompetitors_SameCountry": "Same_Country_All",
        "Mean_AllSimilarityScore": "SimilarityScoreMean_All",
    })
    .with_columns(
        pl.col("N_Competitors_All").fill_null(0),
        pl.col("Same_Country_All").fill_null(0),
        pl.col("SimilarityScoreMean_All").fill_null(0.0),
    )
)

# Sostituisco "Stay" con il valore attuale di "GrowthStageGroup"
df = df.with_columns(
    pl.when(pl.col("GrowthNextStageGroup") == "Stay")
    .then(pl.col("GrowthStageGroup"))
    .otherwise(pl.col("GrowthNextStageGroup"))
    .alias("GrowthNextStageGroup")
)

# Sostituisce CompanyID con interi crescenti (stessa azienda → stesso intero)
company_id_map = (
    df.select("CompanyID").unique()
    .sort("CompanyID")
    .with_row_index(name="_new_id", offset=1)
)
other_cols = [c for c in df.columns if c != "CompanyID"]
df = (
    df.join(company_id_map, on="CompanyID", how="left")
    .drop("CompanyID")
    .rename({"_new_id": "CompanyID"})
    .select(["CompanyID"] + other_cols)
)

print(f'Colonne: {len(df.columns)}') 
print(f'Righe: {len(df)}')

df.write_csv("data/panel.csv.gz", compression="gzip")

Colonne: 122
Righe: 882324


In [ ]:
T=7

lastYear=2024-T

nowindow=False

if(nowindow): #esperimento senza finestra temporale

    # Per ogni CompanyID del dataset che modella la finestratemporale, seleziona la riga con Age massimo
    # dove GrowthStageGroup, GrowthNextStageGroup e Total_People non sono nulli.
    # Se la riga con Age massimo ha valori nulli, prende la precedente valida.
    
    dataset_window = pl.read_csv(config['paths']['dataset_window'],null_values=["NA"])

    ids_prev=dataset_window["CompanyID"].to_list()
    print(len(ids_prev))

    df_no_tw = (
        df
        .sort(["CompanyID", "Age"])
        .filter(
            pl.col("CompanyID").is_in(ids_prev) &
            pl.col("GrowthStageGroup").is_not_null() &
            pl.col("GrowthNextStageGroup").is_not_null() &
            pl.col("Total_People").is_not_null()
        )
        .group_by("CompanyID")
        .agg([
            pl.all().exclude(["TotalRaised_Est", "WorkExp_Idx_Mean", "Highest_Degree_Mean",'Avg_Earliest_Year'])
            .sort_by("Age").last(),
            pl.col("TotalRaised_Est").sum(),
            pl.col("WorkExp_Idx_Mean").mean(),
            pl.col("Highest_Degree_Mean").mean(),
            pl.col("Avg_Earliest_Year").mean(),
        ])
    )

    df_target_final = df_no_tw.rename({"GrowthNextStageGroup": "Target"})

    # Sostituisci i competitor temporalizzati con quelli statici (tutti i competitor, senza filtro annuale)
    df_target_final = (
        df_target_final
        .drop("N_Competitors", "Same_Country", "SimilarityScoreMean")
        .join(nowindow_competitor_stats, on="CompanyID", how="left")
        .join(nowindow_similarity_stats, on="CompanyID", how="left")
        .rename({
            "N_AllCompetitors": "N_Competitors",
            "N_AllCompetitors_SameCountry": "Same_Country",
            "Mean_AllSimilarityScore": "SimilarityScoreMean",
        })
        .with_columns(
            pl.col("N_Competitors").fill_null(0),
            pl.col("Same_Country").fill_null(0),
            pl.col("SimilarityScoreMean").fill_null(0.0),
        )
    )

else:

    df_preseed = df.filter(
        pl.col("GrowthStageGroup")
        .eq("Early")
        .any()
        .over("CompanyID")
    )
    df_preseed=df_preseed.drop_nulls(subset=['GrowthStageGroup'])


    df_preseed = df_preseed.select(["CompanyID", "Age"])

    # aggregazioni per azienda
    df_combined = (
        df_preseed
        .group_by("CompanyID")
        .agg([
            pl.col("Age").min().alias("StartingAge"),
            pl.col("Age").max().alias("LastAge"),
        ])
        .with_columns(
            pl.min_horizontal(
                pl.col("StartingAge") + T,
                pl.col("LastAge")
            ).alias("TargetAge")
        )
        .filter(pl.col("StartingAge") <= 2) # filtro aggiunto per evitare valori troppo diversi (in ogni caso la maggior parte delle aziende aveva un Age compreso tra 0 e 2 al momento dell'entrata in preseed)
    )

    df = df.join(df_combined, on="CompanyID")


    df_target = df.filter(
        pl.col("Age") == pl.col("TargetAge")
    ).select(["CompanyID", "StartingAge" ,"TargetAge" ,'GrowthStageGroup','TimeNextStageGroup','GrowthNextStageGroup'])


    df_target = df_target.with_columns(
        pl.when(pl.col("TargetAge") + pl.col("TimeNextStageGroup")<=pl.col("StartingAge") + T) 
        .then(pl.col("GrowthNextStageGroup"))
        .otherwise(pl.col("GrowthStageGroup"))
        .alias("Target")
    )

    df_target_panel=df.join(df_target.select(["CompanyID", "Target"]),on='CompanyID')


    df_target_final = df_target_panel.filter(
        (pl.col("Age") == pl.col("StartingAge")) &
        (pl.col("YearFounded")+pl.col("Age") <= lastYear) &
        (pl.col("YearFounded")+pl.col("Age") >= 2010)
    )


print(f'Colonne: {len(df_target_final.columns)}') 
print(f'Righe: {len(df_target_final)}')